# OpenLux Hub — 02 Contexte quotidien par canton

La table **`gold_luxembourg_daily_context`** croise, pour chaque jour et chacun des 12 cantons :
- la météo horaire agrégée (modèle Open-Meteo au chef-lieu du canton) ;
- l'indice européen de qualité de l'air (modèle CAMS) ;
- les jours fériés légaux et vacances scolaires luxembourgeois ;
- les nouveaux événements trafic (CITA, Ville) et les chantiers actifs.

Aucune valeur n'est imputée : un indicateur absent reste `NULL`.

## 1. Chargement de `gold_luxembourg_daily_context`

La table consolide chaque jour civil pour chaque commune les indicateurs clés du territoire.

In [ ]:
import duckdb

con = duckdb.connect()

# Contexte quotidien par canton (aucune valeur imputée : NULL si la source manque)
df_context = con.execute("""
    SELECT context_date, canton, avg_temperature_c, total_precipitation_mm, max_european_aqi,
           is_public_holiday, holiday_name, is_school_holiday, new_traffic_events, active_roadworks
    FROM 'data/gold/gold_luxembourg_daily_context.parquet'
    ORDER BY context_date DESC, canton
    LIMIT 12
""").df()

df_context

## 2. Complétude mesurée

Plutôt qu'un « grade » déclaratif, on mesure la part des couples (jour, canton) qui disposent réellement de chaque indicateur.

In [ ]:
# Complétude mesurée : part des jours/cantons disposant de chaque indicateur
df_quality = con.execute("""
    SELECT canton,
           count(*) AS nb_jours,
           round(100 * avg((avg_temperature_c IS NOT NULL)::INT), 1) AS pct_meteo,
           round(100 * avg((max_european_aqi IS NOT NULL)::INT), 1) AS pct_air
    FROM 'data/gold/gold_luxembourg_daily_context.parquet'
    GROUP BY canton ORDER BY canton
""").df()

df_quality

## 3. Grèves dans la presse

`gold_strikes_news` ne conserve que le titre, l'éditeur, la date et le lien des articles mentionnant une grève (règle Factual-Only). Le pays est déduit par mots-clés.

In [ ]:
# Articles de presse mentionnant une grève (titre + lien, pays déduit par mots-clés)
df_strikes = con.execute("""
    SELECT country_code, publisher, title, published_at_utc, url
    FROM 'data/gold/gold_strikes_news.parquet'
    ORDER BY published_at_utc DESC
    LIMIT 10
""").df()

df_strikes

## 4. Occupation des parkings par quartier

Série temporelle accumulée à chaque collecte du flux temps réel de la Ville de Luxembourg.

In [ ]:
# Occupation moyenne des parkings par quartier (Ville de Luxembourg)
df_parkings = con.execute("""
    SELECT district,
           count(DISTINCT parking_id) AS parkings,
           round(avg(occupancy_pct), 1) AS occupation_moyenne_pct
    FROM 'data/gold/gold_mobility_parkings.parquet'
    GROUP BY district ORDER BY occupation_moyenne_pct DESC NULLS LAST
""").df()

df_parkings